In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# Load Healthcare Dataset
# -----------------------------
df = pd.read_csv("heart.csv")

X = df.drop("target", axis=1).values
y = df["target"].values

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

# -----------------------------
# Vertical Data Split (13 features: 7 + 6)
# -----------------------------
# Client A gets first 7 features: age, sex, cp, trestbps, chol, fbs, restecg
XA_train = X_train[:, :7]
XA_test = X_test[:, :7]

# Client B gets remaining 6 features: thalach, exang, oldpeak, slope, ca, thal
XB_train = X_train[:, 7:]
XB_test = X_test[:, 7:]

# -----------------------------
# Client Models (deeper network)
# -----------------------------
class ClientModel(nn.Module):
    def __init__(self, input_size):
        super(ClientModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

    def forward(self, x):
        return self.model(x)

# -----------------------------
# Server Model (deeper network)
# -----------------------------
class ServerModel(nn.Module):
    def __init__(self):
        super(ServerModel, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )

    def forward(self, xa, xb):
        x = torch.cat((xa, xb), dim=1)
        return self.model(x)

# -----------------------------
# Initialize Models
# -----------------------------
clientA = ClientModel(7)
clientB = ClientModel(6)
server = ServerModel()

criterion = nn.BCELoss()

optA = optim.Adam(clientA.parameters(), lr=0.01)
optB = optim.Adam(clientB.parameters(), lr=0.01)
optS = optim.Adam(server.parameters(), lr=0.01)

# -----------------------------
# Training
# -----------------------------
epochs = 100

for epoch in range(epochs):

    # Local embeddings
    embedA = clientA(XA_train)
    embedB = clientB(XB_train)

    # Server prediction
    preds = server(embedA, embedB)

    loss = criterion(preds, y_train)

    optA.zero_grad()
    optB.zero_grad()
    optS.zero_grad()

    loss.backward()

    optA.step()
    optB.step()
    optS.step()

    if (epoch+1) % 10 == 0:
        print("Epoch:", epoch+1, "Loss:", round(loss.item(), 4))

# -----------------------------
# Testing
# -----------------------------
with torch.no_grad():

    embedA = clientA(XA_test)
    embedB = clientB(XB_test)

    preds = server(embedA, embedB)

    predicted = (preds > 0.5).float()

    accuracy = (predicted == y_test).float().mean()

print("Test Accuracy:", round(accuracy.item() * 100, 2), "%")

Epoch: 10 Loss: 0.424
Epoch: 20 Loss: 0.2946
Epoch: 30 Loss: 0.2278
Epoch: 40 Loss: 0.1752
Epoch: 50 Loss: 0.123
Epoch: 60 Loss: 0.0849
Epoch: 70 Loss: 0.0597
Epoch: 80 Loss: 0.03
Epoch: 90 Loss: 0.0146
Epoch: 100 Loss: 0.0095
Test Accuracy: 97.56 %
